In [0]:
%run ./01_logging_utils.py

In [0]:
DDL = {
 T_DIM_COIN: """coin_key BIGINT NOT NULL, coin_id STRING NOT NULL, coin_name STRING,
   is_stablecoin BOOLEAN, first_seen_ts TIMESTAMP, last_seen_ts TIMESTAMP, load_timestamp TIMESTAMP""",
 T_DIM_DATE: """date_key INT NOT NULL, full_date DATE, year INT, quarter INT, month INT,
   month_name STRING, day_name STRING, week_of_year INT, day_of_month INT, day_of_week INT,
   is_weekend BOOLEAN""",
 T_DIM_HOUR: "hour_key INT NOT NULL, hour_label STRING, day_part STRING",
 T_DIM_SCENARIO: "scenario_key BIGINT NOT NULL, scenario_name STRING, description STRING",
 T_BRIDGE: "scenario_key BIGINT NOT NULL, coin_key BIGINT NOT NULL, weight DOUBLE",
 T_FACT_HOURLY: """coin_key BIGINT NOT NULL, event_ts TIMESTAMP NOT NULL, date_key INT, hour_key INT,
   price DOUBLE, market_cap DOUBLE, total_volume DOUBLE, prev_price DOUBLE,
   hourly_return_pct DOUBLE, volume_change_pct DOUBLE, is_volume_spike BOOLEAN,
   volume_to_mcap_ratio DOUBLE, batch_id STRING, load_timestamp TIMESTAMP""",
 T_FACT_DAILY: """coin_key BIGINT NOT NULL, date_key INT NOT NULL, open_price DOUBLE, close_price DOUBLE,
   high_price DOUBLE, low_price DOUBLE, avg_price DOUBLE, avg_market_cap DOUBLE, avg_volume DOUBLE,
   daily_return_pct DOUBLE, intraday_range_pct DOUBLE, volatility_pct DOUBLE,
   hours_count INT, load_timestamp TIMESTAMP""",
 T_FACT_PORTF: """scenario_key BIGINT NOT NULL, date_key INT NOT NULL, portfolio_return_pct DOUBLE,
   market_condition STRING, is_high_volatility BOOLEAN, load_timestamp TIMESTAMP""",
}
CLUSTER = {T_FACT_HOURLY: "coin_key, date_key",
           T_FACT_DAILY:  "coin_key, date_key",
           T_FACT_PORTF:  "date_key"}

rid, t0 = new_batch_id("run"), datetime.now()
for t, cols in DDL.items():
    base = f"CREATE TABLE IF NOT EXISTS {t} ({cols}) USING DELTA"
    try:
        spark.sql(base + (f" CLUSTER BY ({CLUSTER[t]})" if t in CLUSTER else ""))
    except Exception as e:
        print(f"Clustering not available for {t}, creating without it:", str(e)[:120])
        spark.sql(base)
log_run(rid, "gold", f"08_gold_setup: {len(DDL)} tables ensured", t0, datetime.now(), "SUCCESS")

In [0]:
PK = {T_DIM_COIN: ["coin_key"], T_DIM_DATE: ["date_key"], T_DIM_HOUR: ["hour_key"],
      T_DIM_SCENARIO: ["scenario_key"], T_BRIDGE: ["scenario_key", "coin_key"],
      T_FACT_HOURLY: ["coin_key", "event_ts"], T_FACT_DAILY: ["coin_key", "date_key"],
      T_FACT_PORTF: ["scenario_key", "date_key"]}
FK = [(T_FACT_HOURLY, "coin_key", T_DIM_COIN), (T_FACT_HOURLY, "date_key", T_DIM_DATE),
      (T_FACT_HOURLY, "hour_key", T_DIM_HOUR), (T_FACT_DAILY, "coin_key", T_DIM_COIN),
      (T_FACT_DAILY, "date_key", T_DIM_DATE), (T_BRIDGE, "scenario_key", T_DIM_SCENARIO),
      (T_BRIDGE, "coin_key", T_DIM_COIN), (T_FACT_PORTF, "scenario_key", T_DIM_SCENARIO),
      (T_FACT_PORTF, "date_key", T_DIM_DATE)]

for t, cols in PK.items():
    try:
        spark.sql(f"ALTER TABLE {t} ADD CONSTRAINT pk_{t.split('.')[-1]} PRIMARY KEY ({', '.join(cols)})")
    except Exception as e:
        print("PK skipped:", t, str(e)[:90])
for child, col, parent in FK:
    try:
        spark.sql(f"ALTER TABLE {child} ADD CONSTRAINT fk_{child.split('.')[-1]}_{col} "
                  f"FOREIGN KEY ({col}) REFERENCES {parent}({col})")
    except Exception as e:
        print("FK skipped:", child, col, str(e)[:90])

In [0]:
display(spark.sql(f"SHOW TABLES IN {G}"))
for t in DDL:
    d = spark.sql(f"DESCRIBE DETAIL {t}").first()
    print(t.split(".")[-1], "| clustering:", list(d["clusteringColumns"] or []))